# TextNCF family (variant C)

Review-text-enhanced NCF on HotelRec 20-core. This notebook collects the
saved artefacts from `bash scripts/run_text_ncf_all.sh`, prints them in
one place, and walks through the architecture, the ablations, and the
two negative results (sub-rating attention collapse + ensemble
degeneration).

All numbers come from `results/text_ncf*/{test_metrics,rating_metrics}.json`
and the corresponding `results/text_ncf/{ensemble,two_stage}_metrics.json`.


In [1]:
import json
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "results").is_dir():
            return cand
    raise RuntimeError(f"Couldn't find repo root from {start}")

REPO = find_repo_root(Path.cwd())
RES  = REPO / "results"
print("Repo:", REPO)

def load_json(rel):
    return json.loads((REPO / rel).read_text())

def show_dict(d, fmt="{:.4f}"):
    return pd.Series({k: (fmt.format(v) if isinstance(v, float) else v) for k, v in d.items()})


Repo: C:\Datadrive\Hriday\Education\SJSU\2nd Sem\CMPE 256\Term Project\GitHub Repo


## 1. Architecture summary

**Base TextNCF** has two parallel branches:

```
GMF branch  : user_emb ⊙ item_emb               → (B, embed_dim=64)
text branch : Linear(user_text) ⊙ Linear(item_text) → (B, text_proj_dim=64)
fusion       : concat → Linear(128) → ReLU → Dropout
                       → Linear(64)  → ReLU → Dropout
                       → Linear(1)   → score
```

Trained with BPR loss, 4 negatives per positive, batch 256, lr 1e-3,
cosine LR decay, patience-5 early stopping.

Text encoder: **all-MiniLM-L6-v2** (384-dim), frozen, encoded offline by
`scripts/encode_text.py`. User profiles average over training-split
reviews only (no leakage); item profiles average over all splits.

**Multi-Task** adds a parallel rating head and a joint loss
`α·BPR + (1-α)·MSE` with α=0.7.

**Sub-rating** decomposes the score into 6 aspect heads (Service,
Cleanliness, Location, Value, Rooms, Sleep Quality) and combines them
with per-user attention weights. Loss `β·BPR + (1-β)·aspect-MSE`,
β=0.6.


## 2. Headline ranking metrics

In [2]:
baseline_results = load_json("results/baselines/baseline_results_20core.json")

variants = {
    "TextNCF (base)":    "results/text_ncf/test_metrics.json",
    "TextNCF GMF-only":  "results/text_ncf_gmf_only/test_metrics.json",
    "TextNCF text-only": "results/text_ncf_text_only/test_metrics.json",
    "TextNCF Multi-Task":"results/text_ncf_mt/test_metrics.json",
    "TextNCF Sub-rating":"results/text_ncf_subrating/test_metrics.json",
}

rows = {}
for name, res in baseline_results.items():
    rows[name] = res
for name, path in variants.items():
    m = load_json(path)
    rows[name] = {k: v for k, v in m.items() if k.startswith(("HR@", "NDCG@"))}

df = pd.DataFrame(rows).T
df = df[["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"]]
df.style.format("{:.4f}").background_gradient(cmap="Greens", subset=["NDCG@10"])


,HR@5,HR@10,HR@20,NDCG@5,NDCG@10,NDCG@20
Popularity,0.3150,0.4215,0.5538,0.2318,0.2662,0.2995
ItemKNN,0.6835,0.6870,0.7091,0.6082,0.6093,0.6150
GMF,0.5553,0.6685,0.7936,0.4498,0.4863,0.5179
TextNCF (base),0.5688,0.6787,0.7951,0.4702,0.5057,0.5351
TextNCF GMF-only,0.5585,0.6720,0.7930,0.4548,0.4915,0.5221
TextNCF text-only,0.5659,0.6891,0.8183,0.4583,0.4981,0.5308
TextNCF Multi-Task,0.5742,0.6864,0.8031,0.4734,0.5097,0.5392
TextNCF Sub-rating,0.5380,0.6677,0.8023,0.4291,0.4710,0.5050


**Multi-Task wins on NDCG@10** (0.510 vs 0.506 base, +0.7 % rel).
Text-only ablation (no GMF) actually beats GMF-only on every k - the
*text* branch is the load-bearing piece of the model.

ItemKNN's NDCG@10 (0.609) is still ~20 % above the best TextNCF, an
artefact of the 1-vs-99 evaluation: ItemKNN's similarity to a user's
training history is a near-perfect signal when the negatives are
random.

## 3. Calibrated rating metrics

In [3]:
rating_files = {
    "GlobalMean (sanity)":    ("results/baselines/rating_metrics_20core.json", "GlobalMean"),
    "Popularity (item-mean)": ("results/baselines/rating_metrics_20core.json", "Popularity"),
    "ItemKNN (k=20)":         ("results/baselines/rating_metrics_20core.json", "ItemKNN"),
    "TextNCF (base)":         ("results/text_ncf/rating_metrics.json", None),
    "TextNCF Multi-Task":     ("results/text_ncf_mt/rating_metrics.json", None),
    "TextNCF Sub-rating":     ("results/text_ncf_subrating/rating_metrics.json", None),
}

rows = []
for name, (path, sub) in rating_files.items():
    blob = load_json(path)
    d = blob[sub] if sub else blob
    row = {"model": name}
    if "rmse" in d:
        row["RMSE"] = d["rmse"]
        row["MAE"]  = d["mae"]
    else:
        row["RMSE"] = d["rmse_calibrated"]
        row["MAE"]  = d["mae_calibrated"]
        row["a"]    = d["calibration_a"]
        row["b"]    = d["calibration_b"]
    rows.append(row)

pd.DataFrame(rows).set_index("model").style.format({"RMSE": "{:.4f}", "MAE": "{:.4f}",
                                                     "a": "{:.4f}", "b": "{:.4f}"})


,RMSE,MAE,a,b
model,,,,
GlobalMean (sanity),0.9315,0.7048,nan,nan
Popularity (item-mean),0.8685,0.6749,nan,nan
ItemKNN (k=20),0.9590,0.7094,nan,nan
TextNCF (base),0.9306,0.7014,0.0093,4.0941
TextNCF Multi-Task,0.9304,0.7035,0.0128,4.1008
TextNCF Sub-rating,0.9309,0.7047,0.0326,4.0671


**Every BPR-trained ranker calibrates flat.** The slope `a` is in the
0.01-0.03 range, meaning the predicted rating is essentially `b ≈ 4.08`
(the train mean) regardless of the model's score. RMSE settles around
0.93 - Popularity's 0.8685 wins because predicting the item mean is
near-optimal on a 78 %-4-or-5-star dataset.

## 4. Ensemble (TextNCF + GMF + ItemKNN) - degenerated

In [4]:
ens = load_json("results/text_ncf/ensemble_metrics.json")
print("Best weights from 66-combo grid:", ens["best_weights"])
print()
print("Test metrics at best weights:")
display(show_dict(ens["test_metrics"]))


Best weights from 66-combo grid: {'text_ncf': 0.0, 'gmf': 0.0, 'knn': 1.0}

Test metrics at best weights:


HR@5       0.6835
NDCG@5     0.6082
HR@10      0.6870
NDCG@10    0.6093
HR@20      0.7091
NDCG@20    0.6150
dtype: object

Grid search picked `(text=0, gmf=0, knn=1)` - i.e. the ensemble
collapsed to ItemKNN alone. After per-user min-max normalisation the
TextNCF and GMF scores aren't *worse* than ItemKNN, but they're noisier
on the metric the search optimises (validation NDCG@10), so any non-zero
weight on them drags the blend down.

Take-away: a per-variant blend doesn't help on this dataset. The team's
Phase-3 LightGBM meta-ensemble is a better home for combining models
because it can learn non-linear interactions, not just convex weights.

## 5. Two-stage retrieval (KNN→TextNCF) - recall-bound

In [5]:
ts = load_json("results/text_ncf/two_stage_metrics.json")
m  = ts["metrics"]
print(f"Retriever       : {ts['retriever']} (top-{ts['retrieve_k']})")
print(f"Re-ranker       : {ts['ranker']}")
print(f"GT recall@200   : {m['gt_recall@200']:.4f}  ← only 5 % of held-out items make the candidate set")
print()
display(show_dict({k: v for k, v in m.items() if k != 'gt_recall@200'}))


Retriever       : ItemKNN (top-200)
Re-ranker       : TextNCF
GT recall@200   : 0.0500  ← only 5 % of held-out items make the candidate set



HR@5       0.3164
NDCG@5     0.2754
HR@10      0.3858
NDCG@10    0.2977
HR@20      0.4871
NDCG@20    0.3231
dtype: object

ItemKNN's open top-200 retrieval over the full 27 K-item space catches
the held-out test item only **5 %** of the time. With the GT then injected
into the candidate set for fairness, the re-ranker has to fish it out of a
pool that's 99.5 % distractors - and HR@10 drops to 0.39, far below the
1-vs-99 setting where the negatives are random.

This is a useful production-relevance signal: a "fast retriever +
slow re-ranker" pipeline only helps if the retriever's recall is high.
Larger `retrieve_k` (e.g. 2000) or a different retriever (popularity
blend, BM25) is the next experiment.

## 6. Sub-rating attention - collapsed onto Cleanliness

In [6]:
sr = load_json("results/text_ncf_subrating/test_metrics.json")
attn = sr["attention_weights"]
display(pd.DataFrame({"weight": attn}))


,weight
Service,0.000329
Cleanliness,0.998354
Location,0.000329
Value,0.000328
Rooms,0.000329
Sleep Quality,0.000329


The per-user attention network (a `(32,)` user embedding → `Linear(6)` →
softmax) collapsed early in training. It puts ≈ 99.8 % of the weight on
**Cleanliness** for every user (std=0). The shared MLP carries all the
ranking signal; the aspect heads are effectively unused.

Diagnoses to try next time:
1. **Warm-start uniform.** Initialise the attention bias so the softmax
   starts at 1/6 per aspect.
2. **Entropy bonus.** Add `+ λ · H(attention)` to the loss to discourage
   collapse early in training.
3. **Aspect supervision weight.** β=0.6 keeps BPR dominant; pushing β to
   0.3 would force the heads to actually predict each aspect.


## 7. Reproducibility

```bash
# one-shot setup (idempotent)
python scripts/encode_text.py --kcore 20 --device cuda
python scripts/fit_itemknn.py --kcore 20

# all five trainings + ablations + ensemble + two-stage + RMSE
bash scripts/run_text_ncf_all.sh
```

Total wall-clock on RTX 5070 Ti: **~80 min** (text encoding 11 min,
five trainings ≈ 35 min total, ensemble grid search ≈ 25 min, two-stage
≈ 5 min, RMSE ≈ 2 min).